# Days 9-10: Landmark Detection + GeoNames + Wikidata Enrichment

**Goal**:
- Detect landmarks using zero-shot CLIP classification
- Resolve place names to structured location data via GeoNames
- Enrich detected entities with Wikidata SPARQL + Wikipedia summaries

**Note**: GeoNames and Wikidata require API credentials.
Add `GEONAMES_USERNAME` to your Colab Secrets.

**Done checkpoint**:
- `images.csv` has `landmark`, `city`, `country`, `entity_type`, `entity_tags` populated for images where detectable

## 1. Mount Drive + Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
PROJECT_DIR = '/content/drive/MyDrive/Projects/VisualSearchSystem'
LOCAL_DATA_DIR = '/content/data'

os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
print(f'CWD: {os.getcwd()}')

In [ ]:
import zipfile, yaml
from pathlib import Path

# Re-extract all datasets to local disk (image paths in images.csv point to /content/data/raw/...)
with open(f'{PROJECT_DIR}/configs/config.yaml') as f:
    _cfg = yaml.safe_load(f)

for ds in _cfg.get('datasets', []):
    local_dir  = Path(LOCAL_DATA_DIR) / 'raw' / ds['name']
    drive_zips = list((Path(PROJECT_DIR) / 'data' / 'raw' / ds['name']).glob('*.zip'))
    images_present = any(local_dir.rglob('*.jpg'))

    if images_present:
        print(f"[{ds['name']}] Already on local disk.")
    elif drive_zips:
        print(f"[{ds['name']}] Extracting from Drive zip...")
        local_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(drive_zips[0], 'r') as zf:
            zf.extractall(local_dir)
        print(f"[{ds['name']}] Done.")
    else:
        print(f"[{ds['name']}] No zip on Drive — run Day 1 notebook first.")

In [ ]:
!pip install -q torch torchvision transformers Pillow pandas numpy tqdm pyyaml requests wikipedia SPARQLWrapper
print('Done.')

## 2. Configure Credentials

In [ ]:
from google.colab import userdata
import os

try:
    os.environ['GEONAMES_USERNAME'] = userdata.get('GEONAMES_USERNAME')
    print('GeoNames username set from Colab Secrets.')
except Exception:
    print('GEONAMES_USERNAME not in Colab Secrets.')
    print('Set it manually: os.environ["GEONAMES_USERNAME"] = "your_username"')
    # os.environ['GEONAMES_USERNAME'] = 'your_username'

## 3. Load Config

In [ ]:
import yaml
from pathlib import Path

with open(f'{PROJECT_DIR}/configs/config.yaml') as f:
    config = yaml.safe_load(f)

BASE  = Path(PROJECT_DIR)
LOCAL = Path(LOCAL_DATA_DIR)

# Images are on local disk; metadata CSV on Drive
for ds in config['datasets']:
    ds['raw_dir'] = str(LOCAL / 'raw' / ds['name'])
config['paths']['metadata_csv'] = str(BASE / 'data/metadata/images.csv')
config['model']['device']       = 'cuda'

import pandas as pd
df = pd.read_csv(config['paths']['metadata_csv'])
print(f'Dataset: {len(df):,} rows | Columns: {list(df.columns)}')
if 'dataset' in df.columns:
    print(f'\nPer dataset:')
    print(df.groupby('dataset').size().to_string())
    landmark_rows = df[df['dataset'] == 'landmarks'] if 'landmarks' in df['dataset'].values else pd.DataFrame()
    print(f'\nLandmark images available: {len(landmark_rows):,}')

## 4. Test Landmark Detector (Single Image)

In [ ]:
from src.models.clip_encoder import CLIPEncoder
from src.models.landmark_detector import LandmarkDetector
from PIL import Image
from IPython.display import display

encoder = CLIPEncoder(model_name=config['model']['clip_model'], device='cuda')
detector = LandmarkDetector(encoder)

# Test on a few samples
for _, row in df.sample(5).iterrows():
    result = detector.analyze(row['path'])
    print(f'ID: {row["image_id"]} | Category: {row.get("category","")}')
    print(f'  Landmark: {result["landmark"]} (conf: {result["landmark_confidence"]:.3f})')
    print(f'  Scene:    {result["scene_tags"][:3]}')
    display(Image.open(row['path']).resize((150, 150)))
    print()

## 5. Test GeoNames Resolver

In [ ]:
from src.enrichment.geonames import GeoNamesResolver

try:
    resolver = GeoNamesResolver()
    test_places = ['Eiffel Tower', 'New York', 'Taj Mahal', 'Tokyo']
    for place in test_places:
        result = resolver.resolve_place(place)
        print(f'{place:20} -> {result}')
except EnvironmentError as e:
    print(f'GeoNames not configured: {e}')
    print('Add GEONAMES_USERNAME to Colab Secrets to enable location enrichment.')

## 6. Test Wikidata + Wikipedia Enrichment

In [ ]:
from src.enrichment.wikidata import enrich_entity
from src.enrichment.wikipedia import enrich_entity_wikipedia

test_entities = ['Eiffel Tower', 'Taj Mahal', 'Golden Gate Bridge']

for entity in test_entities:
    print(f'\n--- {entity} ---')
    wd = enrich_entity(entity)
    wp = enrich_entity_wikipedia(entity)
    print(f'  Wikidata entity_type: {wd.get("entity_type")}')
    print(f'  Wikidata tags:        {wd.get("entity_tags", [])[:4]}')
    print(f'  Wikipedia summary:    {str(wp.get("entity_description", ""))[:120]}...')
    print(f'  Wikipedia URL:        {wp.get("wikipedia_url")}')

## 7. Run Full Location Enrichment

In [ ]:
# The landmarks dataset contains named world landmarks (Eiffel Tower, Taj Mahal, etc.)
# so GeoNames and Wikidata enrichment will now produce real results.
# Fashion and food images will get scene-level tags where detectable.

from scripts.enrich_locations import enrich_locations
enrich_locations(config, save_every=100)

## 8. Run Entity Enrichment (for images with detected landmarks)

In [ ]:
from scripts.enrich_entities import enrich_entities
enrich_entities(config, save_every=50)

## 9. Inspect Enriched Metadata

In [ ]:
import pandas as pd

df = pd.read_csv(config['paths']['metadata_csv'])
print(f'Columns: {list(df.columns)}')
print(f'\nLandmark distribution (top 10):')
print(df['landmark'].value_counts().head(10))

has_city = df['city'].notna() & (df['city'] != '')
print(f'\nImages with city: {has_city.sum():,}')

has_entity = df['entity_type'].notna() & (df['entity_type'] != '')
print(f'Images with entity_type: {has_entity.sum():,}')

enriched = df[has_entity]
if len(enriched) > 0:
    print('\nSample enriched rows:')
    for _, row in enriched.sample(min(3, len(enriched))).iterrows():
        print(f'  ID:{row["image_id"]} | {row.get("landmark")} | {row.get("entity_type")} | {row.get("city")}, {row.get("country")}')

## ✅ Days 9-10 Done Checkpoint

In [ ]:
import pandas as pd

df = pd.read_csv(config['paths']['metadata_csv'])
assert 'landmark' in df.columns, 'landmark column missing'
assert 'entity_type' in df.columns, 'entity_type column missing'

print('Days 9-10 COMPLETE')
print(f'  Landmark column: populated for {df["landmark"].notna().sum():,} images')
print(f'  Entity type:     populated for {df["entity_type"].notna().sum():,} images')
print(f'  Location data:   populated for {df["city"].notna().sum():,} images')